# LSTM -- Long Short-Term Memory (Pipeline B)

**Model**: LSTM via `nowcast_lstm.LSTM`
**Target**: `gdpc1`
**Variables**: Cat3 (53 + COVID = 56)
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025
**Scaling**: Library handles internally
**HP tuning**: YES -- n_units x n_layers x dropout on validation
**n_timesteps**: 6
**Library notes**: `nowcast_lstm` handles NaN filling, scaling, and lagging natively.
Only `gen_lagged_data` is called from helpers. No `flatten_data`/`mean_fill_dataset`.
Features/obs ratio: 56 vars, 194 quarterly obs -- handled by internal dropout.


In [1]:
import sys, os
import numpy as np
import pandas as pd
from nowcast_lstm.LSTM import LSTM
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams["figure.figsize"] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, gen_vintage_data, get_features, load_data,
                     PREDICTIONS_DIR, TARGET, COVID)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_START = "1959-01-01"
TRAIN_END   = "2007-12-31"
VAL_END     = "2016-12-31"
TEST_START  = "2017-01-01"
TEST_END    = "2025-12-31"
N_TIMESTEPS = 6
VINTAGES = {"m1": -2, "m2": -1, "m3": 0, "post1": 1}

print("LSTM -- Cat3, n_timesteps={}".format(N_TIMESTEPS))


LSTM -- Cat3, n_timesteps=6


In [2]:
monthly, _, metadata = load_data()
features = get_features("cat3", with_covid=True)
print("Features: {}".format(len(features)))

# Phase A: HP tuning on validation window (2008-2016)
# Create a single validation-period dataset
val_data = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= VAL_END)].reset_index(drop=True)
val_data = val_data[["date", TARGET] + features]  # Cat3 vars only
# Use m3 vintage (most data available) for tuning
val_masked = gen_lagged_data(metadata, val_data.copy(), pd.Timestamp(VAL_END), lag=0)

print("Tuning HPs on validation window...")
best_score = np.inf
best_hps = {"n_hidden": 32, "n_layers": 1, "dropout": 0.0}

# Light grid -- 8 combinations
for n_hidden in [32, 64]:
    for n_layers in [1, 2]:
        for dropout in [0.0, 0.2]:
            try:
                model = LSTM(
                    data=val_masked, target_variable=TARGET,
                    n_timesteps=N_TIMESTEPS,
                    fill_na_func=np.nanmean, fill_ragged_edges_func=np.nanmean,
                    n_models=5, train_episodes=50, batch_size=50,
                    decay=0.98, n_hidden=n_hidden, n_layers=n_layers,
                    dropout=dropout,
                    criterion=torch.nn.MSELoss(),
                    optimizer=torch.optim.Adam,
                    optimizer_parameters={"lr": 1e-2, "weight_decay": 0.0},
                )
                model.train(quiet=True)
                preds = model.predict(val_masked)
                # Evaluate on Q-end months only
                q_preds = preds[preds.date.dt.month.isin([3, 6, 9, 12])]
                actual_mask = ~pd.isna(q_preds[TARGET])
                if actual_mask.sum() > 5:
                    score = np.nanmean((q_preds.loc[actual_mask, "predictions"].values -
                                         q_preds.loc[actual_mask, TARGET].values) ** 2)
                    if score < best_score:
                        best_score = score
                        best_hps = {"n_hidden": n_hidden, "n_layers": n_layers, "dropout": dropout}
            except Exception:
                pass

print("Best HPs: n_hidden={}, n_layers={}, dropout={}".format(
    best_hps["n_hidden"], best_hps["n_layers"], best_hps["dropout"]))


Features: 56
Tuning HPs on validation window...


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Best HPs: n_hidden=32, n_layers=1, dropout=0.0


In [3]:
# Phase B: Rolling test loop with frozen HPs
test_quarters = monthly[(monthly["date"] >= TEST_START) &
                         (monthly["date"] <= TEST_END) &
                         monthly["date"].dt.month.isin([3, 6, 9, 12])]["date"].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly["date"] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    base = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= pd_q)].reset_index(drop=True)
    base = base[["date", TARGET] + features]  # Cat3 vars only

    for vn, offset in VINTAGES.items():
        vintage_date = pd_q + pd.DateOffset(months=offset)
        train_m = gen_vintage_data(metadata, base.copy(), pd_q, vintage_date)
        train_m.loc[train_m["date"] == pd_q, TARGET] = np.nan

        try:
            model = LSTM(
                data=train_m, target_variable=TARGET,
                n_timesteps=N_TIMESTEPS,
                fill_na_func=np.nanmean, fill_ragged_edges_func=np.nanmean,
                n_models=10, train_episodes=100, batch_size=50,
                decay=0.98,
                n_hidden=best_hps["n_hidden"],
                n_layers=best_hps["n_layers"],
                dropout=best_hps["dropout"],
                criterion=torch.nn.MSELoss(),
                optimizer=torch.optim.Adam,
                optimizer_parameters={"lr": 1e-2, "weight_decay": 0.0},
            )
            model.train(quiet=True)

            # Predict the GDP target quarter; vintage_date only controls availability.
            pred_df = model.predict(train_m)
            pred_row = pred_df[pred_df["date"] == pd_q]
            if len(pred_row) > 0:
                pred = pred_row["predictions"].values[0]
            else:
                pred = np.nan
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vn].append(pred)

    if (i + 1) % 6 == 0:
        print("  {} / {}".format(i + 1, len(test_quarters)))

print("Done. {} quarters, {} failures.".format(len(test_quarters), failed))


  6 / 36


  12 / 36


  18 / 36


  24 / 36


  30 / 36


  36 / 36
Done. 36 quarters, 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({
        "date": test_quarters, "actual": actuals_list,
        "prediction": predictions[vn],
    }).to_csv(os.path.join(PREDICTIONS_DIR, "lstm_{}.csv".format(vn)), index=False)
    print("Saved lstm_{}.csv".format(vn))


Saved lstm_m1.csv
Saved lstm_m2.csv
Saved lstm_m3.csv
Saved lstm_post1.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m] - p[m]) ** 2)) if m.sum() > 0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m] - p[m])) if m.sum() > 0 else np.nan

panels = {
    "Pre-COVID  (2017-2019)": ("2017-01-01", "2019-12-31"),
    "COVID      (2020-2021)": ("2020-04-01", "2021-12-31"),
    "Post-COVID (2022-2025)": ("2022-01-01", "2025-12-31"),
    "Full test  (2017-2025)": ("2017-01-01", "2025-12-31"),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print("LSTM HPs: n_hidden={}, n_layers={}, dropout={}".format(
    best_hps["n_hidden"], best_hps["n_layers"], best_hps["dropout"]))
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}  N={}".format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))


LSTM HPs: n_hidden=32, n_layers=1, dropout=0.0
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002877  MAE=0.002213  N=12
  m2  RMSFE=0.003155  MAE=0.002430  N=12
  m3  RMSFE=0.003403  MAE=0.002701  N=12
  post1  RMSFE=0.003573  MAE=0.002870  N=12
--- COVID      (2020-2021) ---
  m1  RMSFE=0.042769  MAE=0.026574  N=7
  m2  RMSFE=0.031174  MAE=0.018254  N=7
  m3  RMSFE=0.023461  MAE=0.015818  N=7
  post1  RMSFE=0.020175  MAE=0.014087  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004899  MAE=0.003750  N=16
  m2  RMSFE=0.004836  MAE=0.003949  N=16
  m3  RMSFE=0.004854  MAE=0.003702  N=16
  post1  RMSFE=0.004976  MAE=0.003898  N=16
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019533  MAE=0.008159  N=36
  m2  RMSFE=0.014596  MAE=0.006652  N=36
  m3  RMSFE=0.011576  MAE=0.006214  N=36
  post1  RMSFE=0.010082  MAE=0.005877  N=36
